# 09 · Strongly nonlinear 2D inverse reconstruction — production

**Run All** in your existing TensorFlow environment. The production switch is **ON by default**; no manual edits are required. This adds **six** full-budget neural trainings, not a large sweep:

- 3 methods: plain PINN, Fourier PINN fixed weights, Fourier PINN adaptive weights.
- 2 frozen observation conditions: clean, 5% noisy.
- 1 predeclared training seed per method/condition (2026).

All six share the 961 terminal interior sensors and prescribed boundaries. **Completed matching runs are cached**, and interrupted ones resume. Nothing in the earlier 39-run cohort is modified. The paper figure uses the author's original `jet`/contour-panel conventions, not the newer visual design.

This is a new demonstration; do not describe it as a statistically repeated superiority test. Optional further repetitions should be decided from the results.

In [ ]:
from pathlib import Path
import sys,os,json
import numpy as np
import pandas as pd
from IPython.display import display, Image
ROOT=Path.cwd().resolve()
if not (ROOT/'inverse_study').is_dir() and (ROOT/'heat'/'inverse_study').is_dir(): ROOT=ROOT/'heat'
if not (ROOT/'inverse_study').is_dir(): raise FileNotFoundError('Open inside the heat project folder')
sys.path.insert(0,str(ROOT))
from inverse_study import nonlinear2d as n2, nonlinear2d_figures as plot2
from inverse_study import pinn_nonlinear2d as pinn2
R=ROOT/'results'/'nonlinear2d'
ref=n2.load_reference(R/'reference'/'reference_n97_dt0.005.npz')
p=n2.problem_spec()
v,xg,yg=n2.grid(65)
Xeval=np.column_stack([xg.ravel(),yg.ravel()])
METHODS=('plain','fourier_fixed','fourier_adaptive')
CONDITIONS=(('clean',0.,0),('noisy5',.05,3100))
SEED=2026
RUN_PRODUCTION = os.environ.get('NONLINEAR2D_AUDIT_ONLY','0') != '1'
print('Training enabled:',RUN_PRODUCTION,'| planned full-budget jobs:',len(METHODS)*len(CONDITIONS))
print('Reference:',ref['n'],'nodes, Δt',ref['dt'])

## Confirm the exact observation and constitutive-law contract (no training)

In [ ]:
observations={}
for name,noise,seed in CONDITIONS:
    folder,obs=n2.create_observations(ref,noise=noise,seed=seed,n_obs_grid=33,root=R)
    observations[name]=obs
    config=pinn2.training_config(profile='production',method='plain',training_seed=SEED,eval_per_dim=65)
    info=pinn2.validate_inputs(p,obs,config)
    print(name,info['n_observations'],'sensors; observation hash',info['observation_sha256'][:14])
assert all(len(observations[n]['y_obs'])==961 for n,_,_ in CONDITIONS)
print('No truth is passed in the observation dictionaries.')

## Full-budget training, resume on matching job identity

Model uses the conservative residual $\theta_t-\mathrm{Fo}\nabla\cdot[a(\theta)\nabla\theta]$, not a linearized surrogate. The trainer is a **separate copy** of the previously tested engine with only the new 2D nonlinear law added. The source hash of the old engine is unchanged, so your 39 completed runs are not invalidated.

Full budget per run: 30,000 Adam steps, 4×30 tanh network, 16 frequencies per coordinate in Fourier variants, 60,000 PDE collocation points, 8,000 boundary points, checkpoint/resume.

In [ ]:
completed=[]
if RUN_PRODUCTION:
    for label,noise,seed in CONDITIONS:
        for method in METHODS:
            cfg=pinn2.training_config(profile='production',method=method,training_seed=SEED,
                    feature_seed=1729,collocation_seed=2718,N_r=60000,N_b=8000,
                    eval_per_dim=65,run_id=f'strong2d_{label}_{method}_seed{SEED}',
                    X_eval=Xeval,eval_times=[0.0,0.5,1.0])
            print(f'>>> {label} / {method}')
            output=pinn2.train_inverse(p,observations[label],cfg,R/'neural')
            completed.append(output)
            print('    ',output['status'],output['run_dir'])
else:
    print('Audit mode: no neural training launched.')

## Reconstruct and evaluate the saved real models

The initial reference is loaded **after** training. The initial error is computed without changing the network prediction. For the independent forward replay, the initial trace is projected to the *prescribed exact Dirichlet boundary*; its maximum boundary correction is recorded explicitly. Replays are cached per completed model.

In [ ]:
from scipy.interpolate import RegularGridInterpolator
from inverse_study.common import write_json,read_json
rows=[]
truth=n2.initial(xg,yg)
for run in sorted((R/'neural').glob('*/status.json')):
    folder=run.parent
    status=json.loads(run.read_text())
    if status.get('status')!='completed' or status.get('profile')!='production': continue
    meta=json.loads((folder/'config.json').read_text())
    if meta['problem'].get('law')!='sigmoid20_2d': continue
    label=next((name for name,_,_ in CONDITIONS if name in meta['config'].get('run_id','')),None)
    if label is None: continue
    with np.load(folder/'predictions.npz',allow_pickle=False) as z:
        U0=z['theta_initial'].reshape(65,65)
        X=z['X_eval']
        assert np.allclose(X,Xeval)
    discrepancy=float(n2.relative_l2(U0,truth))
    raw_bdry=np.r_[U0[0,:],U0[-1,:],U0[:,0],U0[:,-1]]
    bdry_corr=float(np.max(np.abs(raw_bdry)))
    eval_file=folder/'evaluation_nonlinear2d.json'
    replay_file=folder/'independent_replay_nonlinear2d.npz'
    if eval_file.exists() and replay_file.exists():
        record=read_json(eval_file)
        if record.get('training_signature') != meta['signature']:
            raise RuntimeError('Existing evaluation has different training signature')
    else:
        projected=U0.copy();projected[[0,-1],:]=0;projected[:,[0,-1]]=0
        replay=n2.forward(65,0.01,spec=p,initial_field=projected)
        ref_here=n2.interpolate(ref,Xeval).reshape(65,65)
        obs=observations[label]
        replay_at_sensors=RegularGridInterpolator((replay['axis'],replay['axis']),replay['terminal'])(obs['X_obs'])
        record=dict(method=meta['config']['method'],condition=label,training_seed=meta['config']['training_seed'],
                    training_signature=meta['signature'],initial_relative_l2=discrepancy,
                    initial_boundary_max_abs_correction=bdry_corr,
                    independent_terminal_relative_l2=n2.relative_l2(replay['terminal'],ref_here),
                    independent_sensor_relative_discrepancy=float(np.linalg.norm(replay_at_sensors-obs['y_obs'])/np.linalg.norm(obs['y_obs'])),
                    replay_grid_nodes=65,replay_dt=0.01,
                    reference_grid_nodes=int(ref['n']),reference_dt=float(ref['dt']),
                    status='computed_from_saved_model_not_oracle_selected')
        np.savez_compressed(replay_file,axis=replay['axis'],theta_initial_projected=projected,theta_terminal=replay['terminal'])
        write_json(eval_file,record)
    rows.append(record|{'run_folder':str(folder.relative_to(ROOT))})
report=R/'report';report.mkdir(exist_ok=True)
if rows:
    result_table=pd.DataFrame(rows).sort_values(['condition','method','training_seed'])
    result_table.to_csv(report/'real_production_results.csv',index=False)
    display(result_table[['condition','method','initial_relative_l2','independent_terminal_relative_l2','initial_boundary_max_abs_correction']])
else:
    print('No completed new 2D model yet; no reconstructed data or fake results plotted.')

## Figure generation belongs to notebook 10, after nonlinear QR exists

Do not generate a reference + three PINNs reconstruction montage. The final experimental comparison should include the mixed nonlinear QR solver as a classical baseline. Notebook 10 prepares reference / QR / Fourier PINN fields with original figure conventions, and numerical tables for all methods.


In [ ]:
print('PINN results evaluated. Run 10_Strongly_nonlinear_2D_QR_production_and_comparison.ipynb to produce a fair QR–PINN figure and summary.')

## Next step

Run **10_Strongly_nonlinear_2D_QR_production_and_comparison.ipynb**. It uses identical frozen observations, solves the true mixed nonlinear QR system, selects gamma without initial truth, and generates the comparison only when computed QR and PINN outputs both exist.
